#Retrieval-Augmented Generation Agent

This notebook showcases a Retrieval-Augmented Generation (RAG) framework that facilitates the processing of natural-language queries from PDF files.

##What is achieved with this project

- PDFs are read and segmented into text chunks
- Text is vectorized and stored in ChromaDB
- Relevant context is extracted through hybrid search (BM25 + vector retrieval)
- Responses are generated based on a lightweight LLM (TinyLLaMA or equivalent)

##Main characteristics

- Hybrid Search (60/40 mix): Combines keyword retrieval and semantics to increase relevance
- Confidence Score (0.0-1.0): Marks answers with low reliability
- Source Grounding: Provides page numbers and text snippets that support answers
- Scalability: Supports multiple PDFs and lengthy documents

##Why Colab?

The framework is developed in Google Colab for leveraging GPUs and simplifying deployment (no local installation required).

##Technological Stack

- Python
- LangChain
- ChromaDB
- BM25 (keyword search)
- Hugging Face / Ollama-compatible LLMs

##Steps

1. Read and parse PDF documents
2. Segment and vectorize text
3. Store vectors in a vector DB
4. Retrieve context (hybrid search)
5. Generate answers based on retrieved information
6. Display answers along with sources and confidence scores

##Goal

Creating an explainable, reliable, and non-hallucinatory question-answering agent backed by credible document evidence.

In [1]:
!pip install chromadb pypdf rank_bm25 requests colorama

In [10]:
!apt-get update
!apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,993 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:12 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:13 https://ppa.launchpadcontent.net/deadsn

This step downloads the required language model using Ollama.

The command !ollama pull tinyllama retrieves the TinyLLaMA model from the Ollama model registry and makes it available for local inference within the notebook environment. This ensures that the model weights are loaded and ready before running the RAG pipeline.

TinyLLaMA is a lightweight, efficient language model suitable for resource-constrained environments like Google Colab, while still providing reasonable performance for question-answering tasks.

In [12]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
import subprocess, time

process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL)

print("Waiting for Ollama to start...")
time.sleep(5)

Waiting for Ollama to start...


In [14]:
!ollama pull tinyllama

In [15]:
import os
import re
import time
import requests
import colorama
from colorama import Fore, Style
import chromadb
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import display

colorama.init(autoreset=True)

#Pipeline Configuration Parameters

This subsection describes the main parameters controlling the pipeline behavior:

 - Model (```MODEL_NAME```): Specifies the LLM used for answer generation (e.g., TinyLLaMA)

 - Chunking (```CHUNK_SIZE```,```CHUNK_OVERLAP```): Controls how PDFs are split into overlapping text chunks for better context retention

 - Retrieval (```TOP_K```): Number of most relevant chunks retrieved for answering queries

 - Hybrid Search Weights (```BM25_WEIGHT```, ```VECTOR_WEIGHT```): Balance between keyword-based (BM25) and semantic (vector) retrieval

 - Generation (```TEMPERATURE```, ```NUM_PREDICT```, ```NUM_CTX```): Controls response creativity, length and context window size


In [16]:
# === Configuration Parameters ===
MODEL_NAME = "tinyllama"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80
TOP_K = 4
BM25_WEIGHT = 0.4
VECTOR_WEIGHT = 0.6
TEMPERATURE = 0.2
NUM_PREDICT = 300
NUM_CTX = 2048


## Global Variables

These variables store and manage data across the RAG pipeline.

 - ```all_chunks``` and ```all_meta``` hold the processed text chunks and their metadata.
  - ```bm25_index``` manages keyword-based retrieval.

 - ```chroma_client``` and ```collection``` handle the vector database for semantic search.

Together, they maintain the system state for document processing and retrieval.

In [17]:
# === Global Variables ===
all_chunks = []
all_meta = []
bm25_index = None
chroma_client = None
collection = None

##Retrieval, Generation, and Analytics Pipeline

This module implements the end-to-end RAG workflow, spanning document ingestion, indexing, retrieval, and response generation.

Documents are parsed and segmented into overlapping chunks, which are indexed using both **BM25 (lexical retrieval)** and **ChromaDB embeddings (semantic retrieval)**. A hybrid ranking function combines both signals via weighted scoring to improve relevance and robustness.

At query time, the system retrieves top-k context, constructs a grounded prompt, and generates responses using the LLM. Outputs are augmented with **page-level citations** and a normalized **confidence score** to improve transparency and reliability.

The pipeline also includes an analytics layer that visualizes retrieval contributions (BM25 vs vector) and estimates **hallucination risk** based on confidence, enabling better interpretability of model behavior.


In [18]:
# Analytics State
last_query_text = ""
last_query_chunks = []
last_query_conf = 0.0

def load_pdf(path):
    global all_chunks, all_meta
    print(f"Extracting text from {path}...")
    reader = PdfReader(path)
    all_chunks = []
    all_meta = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            # Normalize whitespace
            text = re.sub(r'\s+', ' ', text)
            chunk_pages(text, i + 1)

    print(f"Extracted {len(all_chunks)} chunks from {len(reader.pages)} pages.")
    build_indexes()

def chunk_pages(text, page_num):
    global all_chunks, all_meta
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + CHUNK_SIZE
        chunk = text[start:end]
        all_chunks.append(chunk)
        all_meta.append({'page': page_num})
        start += (CHUNK_SIZE - CHUNK_OVERLAP)

def build_indexes():
    global bm25_index, chroma_client, collection
    print("Building indexes (this may take a moment)...")

    # 1. Build BM25
    tokenized_corpus = [chunk.lower().split() for chunk in all_chunks]
    bm25_index = BM25Okapi(tokenized_corpus)

    # 2. Build ChromaDB
    chroma_client = chromadb.Client()
    # Reset if exists
    try:
        chroma_client.delete_collection("rag_docs")
    except:
        pass

    collection = chroma_client.create_collection("rag_docs")

    # Add to Chroma (handles embedding automatically via sentence-transformers)
    ids = [str(i) for i in range(len(all_chunks))]
    collection.add(
        documents=all_chunks,
        metadatas=all_meta,
        ids=ids
    )
    print("RAG engine loaded. Indexes ready.")

def hybrid_search(query):
    # 1. BM25 Search
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)

    if max(bm25_scores) > 0:
        bm25_norm = [float(s) / max(bm25_scores) for s in bm25_scores]
    else:
        bm25_norm = [0.0] * len(bm25_scores)

    # 2. ChromaDB Vector Search (get TOP_K * 2)
    results = collection.query(
        query_texts=[query],
        n_results=min(TOP_K * 2, len(all_chunks)),
        include=["documents", "metadatas", "distances"]
    )

    vector_results = []
    if results['ids'] and len(results['ids'][0]) > 0:
        for idx_str, doc, meta, dist in zip(results['ids'][0], results['documents'][0], results['metadatas'][0], results['distances'][0]):
            idx = int(idx_str)
            vector_score = max(0.0, 1.0 - dist)
            vector_results.append((idx, vector_score, doc, meta))

    # 3. Blend Scores
    final_scores = []
    for idx, v_score, doc, meta in vector_results:
        b_score = bm25_norm[idx]
        blend = (VECTOR_WEIGHT * v_score) + (BM25_WEIGHT * b_score)
        final_scores.append((blend, v_score, b_score, doc, meta['page']))

    # Sort and take TOP_K
    final_scores.sort(key=lambda x: x[0], reverse=True)
    return final_scores[:TOP_K]

def compute_confidence(retrieved):
    if not retrieved:
        return 0.0
    avg_score = sum(score for score, _, _, _, _ in retrieved) / len(retrieved)
    return avg_score

def render_confidence_bar(score):
    bar_len = 20
    filled = int(score * bar_len)
    bar = "█" * filled + "░" * (bar_len - filled)

    if score >= 0.6:
        color = Fore.GREEN
    elif score >= 0.35:
        color = Fore.YELLOW
    else:
        color = Fore.RED

    return f"{color}[{bar}] {score:.2f}{Style.RESET_ALL}"

def build_prompt(query, chunks):
    context_str = ""
    for score, v_score, b_score, doc, page in chunks:
        context_str += f"[Page {page}] {doc}\n\n"

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: 'Not found in the document.'

Context:
{context_str}

Question: {query}
Answer:"""
    return prompt

def answer_query(query):
    global last_query_text, last_query_chunks, last_query_conf
    start_time = time.time()
    print("  Searching (Hybrid: BM25 + Vector)...")

    chunks = hybrid_search(query)
    conf_score = compute_confidence(chunks)

    # Save to global state for /analyze
    last_query_text = query
    last_query_chunks = chunks
    last_query_conf = conf_score

    print(f"  Confidence: {render_confidence_bar(conf_score)}")
    if conf_score < 0.15:
        print(f"  {Fore.RED}Warning: Low confidence match. The answer may not be in the PDF.{Style.RESET_ALL}")

    print("  Generating answer (TinyLLaMA)...")
    prompt = build_prompt(query, chunks)

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": NUM_PREDICT,
            "num_ctx": NUM_CTX
        }
    }

    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload)
        response.raise_for_status()
        result = response.json()['response']
        print(f"\n  {Fore.CYAN}Answer:{Style.RESET_ALL}\n  {result.strip()}\n")
    except Exception as e:
        print(f"  {Fore.RED}Error calling Ollama API: {e}{Style.RESET_ALL}")
        return

    print(f"  {Fore.MAGENTA}Sources Used:{Style.RESET_ALL}")
    pages_referenced = set()
    for i, (score, v_score, b_score, doc, page) in enumerate(chunks):
        preview = doc.replace('\n', ' ')[:120] + "..." if len(doc) > 120 else doc.replace('\n', ' ')
        print(f"  [{i+1}] Page {page} | Score: {render_confidence_bar(score)}")
        print(f"      '{preview}'")
        pages_referenced.add(page)

    print(f"\n  Pages referenced: {sorted(list(pages_referenced))}")
    print(f"  Response time: {time.time() - start_time:.1f}s\n")

def display_research_dashboard():
    if not last_query_chunks:
        print(f"  {Fore.RED}No previous query found to analyze. Please ask a question first.{Style.RESET_ALL}")
        return

    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Research Analytics: '{last_query_text}'", fontsize=16, fontweight='bold', y=1.05)

    # --- Plot 1: Retrieval Mechanism Breakdown ---
    chunk_labels = [f"Chunk {i+1}\n(Pg {c[4]})" for i, c in enumerate(last_query_chunks)]
    v_scores = [c[1] for c in last_query_chunks]
    b_scores = [c[2] for c in last_query_chunks]

    x = np.arange(len(chunk_labels))
    width = 0.35

    rects1 = ax1.bar(x - width/2, v_scores, width, label='Vector Sim (Semantic)', color='#4c72b0')
    rects2 = ax1.bar(x + width/2, b_scores, width, label='BM25 (Keyword)', color='#dd8452')

    ax1.set_ylabel('Normalized Score', fontsize=12)
    ax1.set_title('Hybrid Search Algorithm Contributions', fontsize=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(chunk_labels)
    ax1.set_ylim(0, 1.1)
    ax1.legend()

    # Add values on top of bars
    for rects in [rects1, rects2]:
        for rect in rects:
            height = rect.get_height()
            ax1.annotate(f'{height:.2f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

    # --- Plot 2: Hallucination Risk Gauge ---
    risk_score = max(0.0, 1.0 - last_query_conf)
    colors = ['#e74c3c' if risk_score > 0.65 else '#f1c40f' if risk_score > 0.4 else '#2ecc71', '#ecf0f1']

    wedges, texts = ax2.pie([risk_score, 1.0 - risk_score], startangle=90, colors=colors,
                            wedgeprops=dict(width=0.4, edgecolor='w'))

    # Make it a half-circle (gauge style)
    for w in wedges:
        w.set_theta1(180)
        w.set_theta2(0)

    ax2.set_title('LLM Hallucination Risk Assessment', fontsize=14)

    # Add text in the middle
    risk_level = "HIGH" if risk_score > 0.65 else "MEDIUM" if risk_score > 0.4 else "LOW"
    ax2.text(0, 0, f"{risk_score*100:.1f}%\nRisk: {risk_level}",
             ha='center', va='center', fontsize=18, fontweight='bold')

    plt.tight_layout()
    display(fig)
    plt.close(fig) # Prevent duplicate inline printing


## PDF Upload & Indexing

This step upload a PDF file directly in Colab. The document is automatically processed, chunked and indexed for retrieval, making it ready for querying.


In [19]:
from google.colab import files
import os

print("Please upload a PDF file.")
uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        print(f"\nProcessing '{filename}'...")
        load_pdf(filename)
        print(f"\n{Fore.GREEN}PDF indexed successfully! Ready for queries.{Style.RESET_ALL}")
else:
    print("No file uploaded.")

Please upload a PDF file.


Saving pdf_sample.pdf to pdf_sample (1).pdf

Processing 'pdf_sample (1).pdf'...
Extracting text from pdf_sample (1).pdf...
Extracted 60 chunks from 8 pages.
Building indexes (this may take a moment)...
RAG engine loaded. Indexes ready.

PDF indexed successfully! Ready for queries.


## CLI Interface

This module provides a command-line interface for interacting with the RAG system. Users can submit natural language queries, access system commands (e.g., `/help`, `/info`, `/sources`) and visualize retrieval analytics.

The interface handles query execution, displays generated answers with source citations and enables real-time exploration of model behavior through built-in analysis tools.


In [27]:
def ensure_ollama():
    import subprocess, time, requests
    try:
        requests.get("http://localhost:11434", timeout=2)
    except:
        subprocess.Popen(['ollama', 'serve'])
        time.sleep(8)

ensure_ollama()

In [28]:
def run_cli():
    print(f"\n{Fore.CYAN}{'='*40}")
    print(f"  RAG PDF Question Answering Agent")
    print(f"{'='*40}{Style.RESET_ALL}")
    print("Type /help for commands. Type your question to query the PDF.\n")

    while True:
        try:
            query = input(f"{Fore.GREEN}You > {Style.RESET_ALL}").strip()

            if not query:
                continue

            if query == "/exit":
                print("  Goodbye!")
                break
            elif query == "/help":
                print("  Available Commands:")
                print("    /help    - Print this command list")
                print("    /sources - List all page numbers present in the loaded PDF")
                print("    /info    - Show current parameters and RAG statistics")
                print("    /analyze - Display visual research graphs for the LAST query")
                print("    /exit    - Gracefully exit the CLI loop")
            elif query == "/analyze":
                display_research_dashboard()
            elif query == "/sources":
                if all_meta:
                    pages = sorted(list(set(m['page'] for m in all_meta)))
                    print(f"  PDF has {len(pages)} pages: {pages}")
                else:
                    print("  No PDF loaded.")
            elif query == "/info":
                print(f"  Model: {MODEL_NAME}")
                print(f"  Total Chunks: {len(all_chunks)}")
                print(f"  Chunk Size: {CHUNK_SIZE} chars (Overlap: {CHUNK_OVERLAP})")
                print(f"  Top-K Retrieval: {TOP_K}")
                print(f"  Weights: Vector={VECTOR_WEIGHT}, BM25={BM25_WEIGHT}")
            else:
                if not all_chunks:
                    print(f"  {Fore.RED}Please upload and index a PDF first!{Style.RESET_ALL}")
                else:
                    answer_query(query)

        except KeyboardInterrupt:
            print("\n  Goodbye!")
            break

run_cli()


  RAG PDF Question Answering Agent
Type /help for commands. Type your question to query the PDF.

You > what is migration period
  Searching (Hybrid: BM25 + Vector)...
  Confidence: [██░░░░░░░░░░░░░░░░░░] 0.10
  Generating answer (TinyLLaMA)...

  Answer:
  The context mentions that during the great Migraton Period following the collapse of the Western Roman Empire, Germanic tribes living in Bohemia moved westwards and settled in Central Bohemia. The population exceeded 100,000 in 1848, but the revolutions in Europe in 1848 also had a negative impact on Prague's population.

  Sources Used:
  [1] Page 1 | Score: [████████░░░░░░░░░░░░] 0.40
      'map drawn by Roman geographer Ptolemaios mentioned a Germanic city called Casurgis . [23] In the late 5th century AD, du...'
  [2] Page 5 | Score: [░░░░░░░░░░░░░░░░░░░░] 0.00
      'uburb, Karlín, was created in 1817, and twenty years later the population exceeded 100,000. The revolutions in Europe in...'
  [3] Page 5 | Score: [░░░░░░░░░░░░░░